# LLM2CLIP — train on Colab (data + models from Hugging Face)

Fine-tunes [microsoft/LLM2CLIP](https://github.com/microsoft/LLM2CLIP) on any Hugging Face image–caption dataset, then downloads the **best checkpoint** to your computer.

**What happens, same recipe as the repo (`llm2clip/run.sh` + `data/extract_embedding.py`):**
1. Captions are embedded once by Microsoft's caption-contrastive LLM (frozen) → cached on disk.
2. The EVA02-CLIP-L/14-336 vision tower + LLM adapter (`TextProj`) are trained with the CLIP contrastive loss against those embeddings.
3. After every epoch it runs image↔text retrieval on a held-out split and keeps the best weights (`best_model.pt`).

| Preset | LLM text encoder | CLIP init | GPU |
|---|---|---|---|
| `llama3.2-1b` (default) | `microsoft/LLM2CLIP-Llama-3.2-1B-Instruct-CC-Finetuned` | `microsoft/LLM2CLIP-Llama3.2-1B-EVA02-L-14-336` | T4 16 GB works |
| `llama3-8b` (paper) | `microsoft/LLM2CLIP-Llama-3-8B-Instruct-CC-Finetuned` | `microsoft/LLM2CLIP-EVA02-L-14-336` | L4 / A100 |

`INIT_FROM="eva_clip"` instead starts from the plain EVA02-CLIP vision weights with a fresh adapter (the paper's training from scratch, needs much more data).

**Runtime → Change runtime type → GPU**, then *Runtime → Run all*. Set `SAVE_TO_DRIVE=True` if you want the run to survive a disconnect.

In [ ]:
#@title 1 · Configuration { display-mode: "form" }
# ── Model ────────────────────────────────────────────────────────────────
PRESET    = "llama3.2-1b"   #@param ["llama3.2-1b", "llama3-8b"]
INIT_FROM = "llm2clip"      #@param ["llm2clip", "eva_clip"]

# ── Dataset (any HF dataset with an image column + caption column(s)) ──
DATASET_ID     = "lmms-lab/flickr30k"  #@param {type:"string"}
DATASET_CONFIG = ""                    #@param {type:"string"}
DATASET_SPLIT  = "test"                #@param {type:"string"}
IMAGE_COLUMN   = "image"               #@param {type:"string"}
CAPTION_COLUMN = "caption"             #@param {type:"string"}
# ^ a string or list-of-strings column; comma-separate several, e.g. "short_caption,long_caption"
MAX_SAMPLES    = 0      #@param {type:"integer"}
# ^ 0 = all rows. Try 2000 for a quick end-to-end test.
VAL_SIZE       = 1000   #@param {type:"integer"}
MAX_CAPTIONS_PER_IMAGE = 5  #@param {type:"integer"}

# ── Training (defaults follow llm2clip/run.sh where it makes sense) ──────
EPOCHS        = 3       #@param {type:"integer"}
BATCH_SIZE    = 0       #@param {type:"integer"}
# ^ 0 = auto from GPU memory (T4 → 32, L4 → 64, A100-40G → 128, 80G → 256)
VISUAL_LR     = 1e-5    #@param {type:"number"}
TEXT_LR       = 0.0     #@param {type:"number"}
# ^ adapter LR. 0 = auto: 1e-5 when INIT_FROM="llm2clip", 1e-4 when "eva_clip" (fresh adapter)
WEIGHT_DECAY  = 0.05    #@param {type:"number"}
VISUAL_LAYER_DECAY = 0.85  #@param {type:"number"}
WARMUP_STEPS  = 100     #@param {type:"integer"}
GRAD_CLIP     = 5.0     #@param {type:"number"}
TRAIN_LAST_N_VISION_BLOCKS = -1  #@param {type:"integer"}
# ^ -1 = train the whole vision tower, 0 = only vision head + adapter (fast, low memory), N = last N blocks
GRAD_CHECKPOINTING = True  #@param {type:"boolean"}
NUM_WORKERS   = 2       #@param {type:"integer"}
SEED          = 42      #@param {type:"integer"}

# ── Output ───────────────────────────────────────────────────────────────
RUN_NAME      = "llm2clip_flickr30k"  #@param {type:"string"}
SAVE_TO_DRIVE = False   #@param {type:"boolean"}
# ^ mounts Google Drive and writes cache + checkpoints there (survives disconnects)
RESUME        = True    #@param {type:"boolean"}
# ^ when SAVE_TO_DRIVE: also keep last_checkpoint.pt (~4.5 GB with optimizer) and resume from it
DOWNLOAD_BEST = True    #@param {type:"boolean"}

In [ ]:
#@title 2 · Install dependencies & clone the repo
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install timm einops ftfy regex braceexpand peft datasets huggingface_hub
!test -d /content/LLM2CLIP || git clone -q --depth 1 https://github.com/microsoft/LLM2CLIP.git /content/LLM2CLIP
!ls /content/LLM2CLIP/llm2clip

Helper module (LLM caption encoder, model loading from the Hub, retrieval metrics). It is also packed into the final zip so you can load the checkpoint locally.

In [ ]:
%%writefile /content/llm2clip_utils.py
"""Helpers for training microsoft/LLM2CLIP on a single GPU (Colab).

Written by the Colab notebook; also shipped inside the downloaded zip so the
best checkpoint can be loaded on your own machine.
"""
import io
import math
import os
import sys
import types

import torch
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# Presets: which LLM text encoder goes with which CLIP checkpoint / config
# ---------------------------------------------------------------------------
PRESETS = {
    # Fits a free T4 (16 GB). 2048-d caption embeddings.
    "llama3.2-1b": dict(
        llm_id="microsoft/LLM2CLIP-Llama-3.2-1B-Instruct-CC-Finetuned",
        llm_name_hint="meta-llama/Llama-3.2-1B-Instruct",
        llm_dim=2048,
        clip_cfg="Llama3.2-1B-EVA02-L-14-336",
        clip_repo="microsoft/LLM2CLIP-Llama3.2-1B-EVA02-L-14-336",
    ),
    # Paper setting. Needs bf16 + >=24 GB for the LLM (L4 / A100). 4096-d embeddings.
    "llama3-8b": dict(
        llm_id="microsoft/LLM2CLIP-Llama-3-8B-Instruct-CC-Finetuned",
        llm_name_hint="meta-llama/Meta-Llama-3-8B-Instruct",
        llm_dim=4096,
        clip_cfg="EVA02-CLIP-L-14-336",
        clip_repo="microsoft/LLM2CLIP-EVA02-L-14-336",
    ),
}
# Vision-only starting point used by the repo's run.sh (--pretrained=eva_clip)
EVA_CLIP_REPO = "QuanSun/EVA-CLIP"
EVA_CLIP_FILE = "EVA02_CLIP_L_336_psz14_s6B.pt"


# ---------------------------------------------------------------------------
# Repo import shims
# ---------------------------------------------------------------------------
def install_shims(code_dir):
    """Make microsoft/LLM2CLIP's `eva_clip` package importable on a stock Colab runtime.

    * xformers: the EVA02 configs use `xattn: true`. Installing xformers on Colab usually
      drags in a different torch build, so if it is missing we route
      `xformers.ops.memory_efficient_attention` to PyTorch's fused SDPA (same math).
      (The repo's non-xattn path also breaks under grad checkpointing, so we keep xattn on.)
    * llm2vec: `eva_clip/__init__` imports a text tower that runs the LLM online. We feed
      pre-computed LLM embeddings instead (exactly like the repo's training), so a stub
      is enough and we avoid llm2vec's pinned transformers version.
    """
    if code_dir not in sys.path:
        sys.path.insert(0, code_dir)
    os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")  # repo uses plain torch.load

    try:
        import xformers.ops  # noqa: F401
        have_xformers = not getattr(sys.modules["xformers"], "_is_shim", False)
    except Exception:
        have_xformers = False

        def memory_efficient_attention(q, k, v, attn_bias=None, p=0.0, scale=None, **_):
            # xformers layout (B, N, H, D) -> SDPA layout (B, H, N, D)
            q, k, v = (t.transpose(1, 2) for t in (q, k, v))
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias, dropout_p=p, scale=scale)
            return out.transpose(1, 2)

        xf = types.ModuleType("xformers")
        xf._is_shim = True
        xops = types.ModuleType("xformers.ops")
        xops.memory_efficient_attention = memory_efficient_attention
        xf.ops = xops
        sys.modules["xformers"] = xf
        sys.modules["xformers.ops"] = xops

    if "llm2vec" not in sys.modules:
        stub = types.ModuleType("llm2vec")

        class LLM2Vec:  # never instantiated here
            def __init__(self, *a, **k):
                raise RuntimeError("Online LLM text tower not used in this notebook.")

        stub.LLM2Vec = LLM2Vec
        sys.modules["llm2vec"] = stub
    return have_xformers


# ---------------------------------------------------------------------------
# LLM caption encoder (same output as llm2vec.LLM2Vec.encode used by the repo)
# ---------------------------------------------------------------------------
class LLM2CLIPTextEncoder:
    """Mean-pooled caption embeddings from an LLM2CLIP caption-contrastive LLM.

    Mirrors `LLM2Vec(model, tok, pooling_mode="mean", max_length=512).encode(captions)`
    from llm2clip/data/extract_embedding.py:
      * Llama-3 Instruct chat wrapping: <|start_header_id|>user<|end_header_id|>\\n\\n{caption}<|eot_id|>
      * left padding with the EOS token
      * mean over the caption tokens (+<|eot_id|>) only, i.e. skip_instruction=True
    """

    HEADER = "<|start_header_id|>user<|end_header_id|>\n\n"
    EOT = "<|eot_id|>"

    def __init__(self, model_id, device="cuda", dtype=torch.bfloat16, max_length=512,
                 model=None, tokenizer=None):
        from transformers import AutoConfig, AutoModel, AutoTokenizer

        self.device = device
        self.max_length = max_length
        self.tokenizer = tokenizer or AutoTokenizer.from_pretrained(model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        if model is None:
            config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
            model = AutoModel.from_pretrained(model_id, config=config, torch_dtype=dtype,
                                              trust_remote_code=True)
        self.model = model.to(device).eval()

    @torch.no_grad()
    def encode(self, captions, batch_size=64, progress=True, out_dtype=torch.float16):
        from tqdm.auto import tqdm

        captions = [c.strip() for c in captions]
        order = sorted(range(len(captions)), key=lambda i: -len(captions[i]))  # less padding
        out = None
        it = range(0, len(order), batch_size)
        for s in tqdm(it, desc="LLM caption embeddings", disable=not progress):
            idx = order[s:s + batch_size]
            caps = [captions[i] for i in idx]
            full = [self.HEADER + c + self.EOT for c in caps]
            enc = self.tokenizer(full, return_tensors="pt", padding=True, truncation=True,
                                 max_length=self.max_length).to(self.device)
            doc_ids = self.tokenizer([c + self.EOT for c in caps], add_special_tokens=False,
                                     truncation=True, max_length=self.max_length)["input_ids"]
            res = self.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
            hidden = res.last_hidden_state if hasattr(res, "last_hidden_state") else res[0]
            emb = torch.stack([hidden[b, -len(d):].float().mean(0) for b, d in enumerate(doc_ids)])
            if out is None:
                out = torch.empty(len(captions), emb.shape[-1], dtype=out_dtype)
            out[torch.tensor(idx)] = emb.to(out_dtype).cpu()
        return out


# ---------------------------------------------------------------------------
# CLIP model (repo's eva_clip code) + weights from the Hugging Face Hub
# ---------------------------------------------------------------------------
def _unwrap_state_dict(sd):
    for key in ("state_dict", "model", "module"):
        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
            sd = sd[key]
    if next(iter(sd)).startswith("module."):
        sd = {k[len("module."):]: v for k, v in sd.items()}
    return sd


def _first_pt_file(repo_id):
    from huggingface_hub import list_repo_files

    files = [f for f in list_repo_files(repo_id) if f.endswith((".pt", ".pth", ".bin"))]
    if not files:
        raise FileNotFoundError(f"No .pt checkpoint found in {repo_id}")
    return sorted(files, key=lambda f: (not f.endswith(".pt"), len(f)))[0]


def build_clip_model(preset, init_from="llm2clip", checkpoint_path=None):
    """Returns (model, preprocess_train, preprocess_val).

    init_from:
      "llm2clip" -> Microsoft's released LLM2CLIP weights (vision tower + LLM adapter)
      "eva_clip" -> EVA02-CLIP-L/14-336 vision weights + freshly initialised adapter (paper recipe)
      "local"    -> a state_dict you saved (e.g. best_model.pt from this notebook)
    """
    from huggingface_hub import hf_hub_download
    from eva_clip import create_model_and_transforms

    p = PRESETS[preset]
    if init_from == "eva_clip":
        ckpt = hf_hub_download(EVA_CLIP_REPO, EVA_CLIP_FILE)
        # repo loader keeps visual.*, drops text.* and visual.head (re-initialised)
        return create_model_and_transforms(p["clip_cfg"], pretrained=ckpt, force_custom_clip=True)

    model, pre_train, pre_val = create_model_and_transforms(p["clip_cfg"], force_custom_clip=True)
    if init_from == "llm2clip":
        checkpoint_path = hf_hub_download(p["clip_repo"], _first_pt_file(p["clip_repo"]))
    elif init_from != "local":
        raise ValueError(f"unknown init_from={init_from!r}")
    sd = _unwrap_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=False))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [k for k in missing if "freqs_" not in k]  # rope tables are rebuilt by the model
    if missing or unexpected:
        print(f"[warn] missing keys: {missing[:8]}{'...' if len(missing) > 8 else ''}")
        print(f"[warn] unexpected keys: {unexpected[:8]}{'...' if len(unexpected) > 8 else ''}")
    else:
        print(f"Loaded all weights from {os.path.basename(checkpoint_path)}")
    return model, pre_train, pre_val


def vision_layer_id(name, depth):
    """Layer index used for layer-wise LR decay (same grouping as training/optim.py)."""
    if name.startswith(("patch_embed", "cls_token", "pos_embed", "mask_token")):
        return 0
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1
    return depth + 1  # norm / fc_norm / head


def configure_trainable(model, train_last_n_vision_blocks=-1):
    """-1: whole vision tower trains; 0: only vision head + adapter; N: last N blocks too."""
    depth = len(model.visual.blocks)
    for name, prm in model.visual.named_parameters():
        lid = vision_layer_id(name, depth)
        if train_last_n_vision_blocks < 0:
            prm.requires_grad = True
        elif lid == depth + 1:
            prm.requires_grad = True
        elif lid == 0:
            prm.requires_grad = train_last_n_vision_blocks >= depth
        else:
            prm.requires_grad = lid > depth - train_last_n_vision_blocks
    for prm in model.text.parameters():
        prm.requires_grad = True
    model.logit_scale.requires_grad = True


def param_groups(model, visual_lr, text_lr, weight_decay, visual_layer_decay=1.0):
    depth = len(model.visual.blocks)
    groups = {}

    def add(prm, lr, wd, key):
        g = groups.setdefault(key, {"params": [], "lr": lr, "weight_decay": wd})
        g["params"].append(prm)

    skip_wd = {"pos_embed", "cls_token"}
    for name, prm in model.visual.named_parameters():
        if not prm.requires_grad:
            continue
        lid = vision_layer_id(name, depth)
        scale = visual_layer_decay ** (depth + 1 - lid)
        no_wd = prm.ndim < 2 or name.endswith(".bias") or name in skip_wd
        add(prm, visual_lr * scale, 0.0 if no_wd else weight_decay, ("v", lid, no_wd))
    for name, prm in model.text.named_parameters():
        if not prm.requires_grad:
            continue
        no_wd = prm.ndim < 2 or name.endswith(".bias")
        add(prm, text_lr, 0.0 if no_wd else weight_decay, ("t", no_wd))
    add(model.logit_scale, text_lr, 0.0, ("scale",))
    return list(groups.values())


def cosine_with_warmup(warmup_steps, total_steps):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    return f


def save_weights(model, path):
    """fp16 parameters + fp32 buffers: same format as Microsoft's released .pt files."""
    params = {n for n, _ in model.named_parameters()}
    sd = {k: (v.half() if k in params and v.is_floating_point() else v).detach().cpu()
          for k, v in model.state_dict().items()}
    torch.save(sd, path)


# ---------------------------------------------------------------------------
# Data + retrieval metrics
# ---------------------------------------------------------------------------
def load_pil(x):
    from PIL import Image

    if isinstance(x, Image.Image):
        img = x
    elif isinstance(x, dict) and x.get("bytes"):
        img = Image.open(io.BytesIO(x["bytes"]))
    elif isinstance(x, dict) and x.get("path"):
        img = Image.open(x["path"])
    elif isinstance(x, (bytes, bytearray)):
        img = Image.open(io.BytesIO(x))
    elif isinstance(x, str):
        img = Image.open(x)
    else:
        raise TypeError(f"Unsupported image type: {type(x)}")
    return img.convert("RGB")


@torch.no_grad()
def retrieval_metrics(img_feats, txt_feats, txt2img, chunk=1024):
    """img_feats (M,E), txt_feats (C,E) L2-normalised; txt2img (C,) owner image of each caption.
    Image->text uses the best-ranked of an image's captions (standard Flickr/COCO protocol)."""
    M, C = img_feats.shape[0], txt_feats.shape[0]
    dev = img_feats.device
    txt2img = txt2img.to(dev)

    # text -> image
    t2i_rank = torch.empty(C, dtype=torch.long, device=dev)
    for s in range(0, C, chunk):
        sim = txt_feats[s:s + chunk] @ img_feats.T
        gt = sim.gather(1, txt2img[s:s + chunk, None])
        t2i_rank[s:s + chunk] = (sim > gt).sum(1)

    # image -> text
    counts = torch.bincount(txt2img, minlength=M)
    kmax = int(counts.max())
    order = torch.argsort(txt2img, stable=True)
    starts = torch.cumsum(counts, 0) - counts
    slot = torch.arange(C, device=dev) - starts[txt2img[order]]
    img2txt = torch.full((M, kmax), -1, dtype=torch.long, device=dev)
    img2txt[txt2img[order], slot] = order
    i2t_rank = torch.empty(M, dtype=torch.long, device=dev)
    for s in range(0, M, chunk):
        sim = img_feats[s:s + chunk] @ txt_feats.T
        idx = img2txt[s:s + chunk]
        gt = sim.gather(1, idx.clamp(min=0)).masked_fill(idx < 0, float("-inf")).max(1, keepdim=True).values
        i2t_rank[s:s + chunk] = (sim > gt).sum(1)

    out = {}
    for k in (1, 5, 10):
        out[f"i2t_R@{k}"] = (i2t_rank < k).float().mean().item() * 100
        out[f"t2i_R@{k}"] = (t2i_rank < k).float().mean().item() * 100
    out["mean_R@1"] = (out["i2t_R@1"] + out["t2i_R@1"]) / 2
    out["rsum"] = sum(v for k, v in out.items() if "_R@" in k and k != "mean_R@1")
    return out

In [ ]:
#@title 3 · Setup (imports, GPU, output folders, optional HF login)
import os, sys, gc, json, math, time, random, hashlib, zipfile
import numpy as np, torch

os.chdir("/content")
sys.path.insert(0, "/content")
import llm2clip_utils as U

REPO_DIR = "/content/LLM2CLIP"
have_xformers = U.install_shims(os.path.join(REPO_DIR, "llm2clip"))
from eva_clip import ClipLoss  # repo code (also validates the shims)

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
device = "cuda"
gpu = torch.cuda.get_device_properties(0)
GPU_GB = gpu.total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(f"GPU: {gpu.name} ({GPU_GB:.0f} GB) | AMP: {AMP_DTYPE} | xformers: {have_xformers} (SDPA fallback otherwise)")

P = U.PRESETS[PRESET]
if PRESET == "llama3-8b" and (GPU_GB < 22 or not BF16):
    print("⚠️  The 8B LLM needs bf16 and ~22 GB for caption embedding. Use an L4/A100 or PRESET='llama3.2-1b'.")
if not BATCH_SIZE:
    BATCH_SIZE = 32 if GPU_GB < 20 else 64 if GPU_GB < 30 else 128 if GPU_GB < 50 else 256
if not TEXT_LR:
    TEXT_LR = 1e-5 if INIT_FROM == "llm2clip" else 1e-4
print(f"Batch size {BATCH_SIZE} | visual LR {VISUAL_LR} | adapter LR {TEXT_LR}")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/llm2clip_runs"
else:
    BASE_DIR = "/content/llm2clip_runs"
OUT_DIR = os.path.join(BASE_DIR, RUN_NAME)
CACHE_DIR = os.path.join(BASE_DIR, "cache")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
print("Outputs →", OUT_DIR)

# Optional: add a Colab secret named HF_TOKEN (key icon on the left) for gated/private repos or faster downloads
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
except Exception:
    _tok = os.environ.get("HF_TOKEN")
if _tok:
    from huggingface_hub import login
    login(_tok); print("Logged in to Hugging Face")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
#@title 4 · Load the dataset from Hugging Face
from datasets import load_dataset

ds = load_dataset(DATASET_ID, DATASET_CONFIG or None, split=DATASET_SPLIT)
print(ds)
cap_cols = [c.strip() for c in CAPTION_COLUMN.split(",") if c.strip()]
for c in [IMAGE_COLUMN] + cap_cols:
    assert c in ds.column_names, f"Column {c!r} not in {ds.column_names}"

if MAX_SAMPLES and MAX_SAMPLES < len(ds):
    ds = ds.shuffle(seed=SEED).select(range(MAX_SAMPLES))

def as_list(x):
    if x is None: return []
    if isinstance(x, str): x = [x]
    return [str(c).strip() for c in x if c is not None and str(c).strip()]

cols = [ds[c] for c in cap_cols]  # reads caption columns only (no image decoding)
captions = [sum((as_list(col[i]) for col in cols), [])[:MAX_CAPTIONS_PER_IMAGE] for i in range(len(ds))]
keep = [i for i, c in enumerate(captions) if c]
if len(keep) < len(ds):
    print(f"Dropping {len(ds) - len(keep)} rows without captions")
    ds = ds.select(keep); captions = [captions[i] for i in keep]

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(ds))
n_val = min(VAL_SIZE, len(ds) // 5)
val_idx, train_idx = sorted(perm[:n_val].tolist()), sorted(perm[n_val:].tolist())
train_ds = ds.select(train_idx).select_columns([IMAGE_COLUMN])
val_ds = ds.select(val_idx).select_columns([IMAGE_COLUMN])
train_caps = [captions[i] for i in train_idx]
val_caps = [captions[i] for i in val_idx]
print(f"train images: {len(train_ds):,} ({sum(map(len, train_caps)):,} captions) | "
      f"val images: {len(val_ds):,} ({sum(map(len, val_caps)):,} captions)")

import matplotlib.pyplot as plt
plt.figure(figsize=(4, 4)); plt.imshow(U.load_pil(train_ds[0][IMAGE_COLUMN])); plt.axis("off"); plt.show()
print("\n".join("• " + c for c in train_caps[0]))

Captions are embedded **once** with the frozen LLM (this is the repo's `extract_embedding.py` step) and cached, so re-runs skip straight to training. The LLM is freed from GPU memory afterwards.

In [ ]:
#@title 5 · Embed captions with the LLM2CLIP LLM (frozen, cached)
def flatten(caps_per_image):
    flat, owner = [], []
    for i, caps in enumerate(caps_per_image):
        flat += caps; owner += [i] * len(caps)
    counts = torch.tensor([len(c) for c in caps_per_image])
    offsets = torch.cumsum(counts, 0) - counts
    return flat, torch.tensor(owner), offsets, counts

train_flat, train_owner, train_off, train_cnt = flatten(train_caps)
val_flat, val_owner, val_off, val_cnt = flatten(val_caps)

key = hashlib.md5(json.dumps([PRESET, train_flat, val_flat]).encode()).hexdigest()[:12]
cache_file = os.path.join(CACHE_DIR, f"caption_emb_{PRESET}_{key}.pt")

if os.path.exists(cache_file):
    cache = torch.load(cache_file)
    train_emb, val_emb = cache["train"], cache["val"]
    print("Loaded cached embeddings:", cache_file)
else:
    llm_dtype = torch.bfloat16 if BF16 else (torch.float32 if PRESET == "llama3.2-1b" else torch.float16)
    try:
        enc = U.LLM2CLIPTextEncoder(P["llm_id"], device=device, dtype=llm_dtype)
    except Exception as e:
        raise RuntimeError(
            f"Could not load {P['llm_id']} ({e}).\nIf this is a transformers-version error, run "
            "`!pip -q install 'transformers==4.44.2'`, restart the runtime and run all cells again."
        ) from e
    bs = 128 if GPU_GB > 30 else 64
    t0 = time.time()
    train_emb = enc.encode(train_flat, batch_size=bs)
    val_emb = enc.encode(val_flat, batch_size=bs)
    print(f"Embedded {len(train_flat) + len(val_flat):,} captions in {time.time() - t0:.0f}s")
    del enc; gc.collect(); torch.cuda.empty_cache()
    torch.save({"train": train_emb, "val": val_emb}, cache_file)

assert train_emb.shape[1] == P["llm_dim"], f"LLM dim {train_emb.shape[1]} != {P['llm_dim']}"
assert torch.isfinite(train_emb.float()).all(), "NaN/inf in caption embeddings"
print("train_emb", tuple(train_emb.shape), "| val_emb", tuple(val_emb.shape))

In [ ]:
#@title 6 · Build the CLIP model (repo's eva_clip) and load weights from Hugging Face
model, preprocess_train, preprocess_val = U.build_clip_model(PRESET, INIT_FROM)
U.configure_trainable(model, TRAIN_LAST_N_VISION_BLOCKS)
model = model.to(device)
if GRAD_CHECKPOINTING:
    model.set_grad_checkpointing(True)

n_all = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{P['clip_cfg']}: {n_all/1e6:.0f}M params, {n_train/1e6:.0f}M trainable "
      f"(vision {sum(p.numel() for p in model.visual.parameters())/1e6:.0f}M, "
      f"adapter {sum(p.numel() for p in model.text.parameters())/1e6:.0f}M)")

In [ ]:
#@title 7 · Data loaders
from torch.utils.data import Dataset, DataLoader

class ImageCaptionEmb(Dataset):
    """image from the HF dataset + one of its pre-computed caption embeddings (random each epoch,
    like the repo's random.choice between short/long captions)."""
    def __init__(self, hf_ds, emb, offsets, counts, transform, train):
        self.ds, self.emb, self.off, self.cnt = hf_ds, emb, offsets, counts
        self.transform, self.train = transform, train
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        try:
            img = U.load_pil(self.ds[i][IMAGE_COLUMN])
        except Exception as e:  # corrupt image → use a neighbour (train only)
            if not self.train: raise
            print("bad image", i, e); return self[(i + 1) % len(self)]
        j = random.randrange(int(self.cnt[i])) if self.train else 0
        return self.transform(img), self.emb[int(self.off[i]) + j].float()

train_set = ImageCaptionEmb(train_ds, train_emb, train_off, train_cnt, preprocess_train, train=True)
val_set = ImageCaptionEmb(val_ds, val_emb, val_off, val_cnt, preprocess_val, train=False)
loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kw)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE * 2, shuffle=False, **loader_kw)
assert len(train_loader) > 0, "Fewer training images than one batch: lower BATCH_SIZE or raise MAX_SAMPLES"
print(f"{len(train_loader)} steps/epoch")

@torch.no_grad()
def evaluate(model):
    model.eval()
    img_feats = []
    for x, _ in val_loader:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            img_feats.append(model.encode_image(x.to(device, non_blocking=True), normalize=True).float())
    img_feats = torch.cat(img_feats)
    txt_feats = []
    for s in range(0, len(val_emb), 1024):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            txt_feats.append(model.encode_text(val_emb[s:s + 1024].to(device).float(), normalize=True).float())
    txt_feats = torch.cat(txt_feats)
    return U.retrieval_metrics(img_feats, txt_feats, val_owner)

In [ ]:
#@title 8 · Train (keeps the best checkpoint by mean R@1 on the val split)
from tqdm.auto import tqdm

optimizer = torch.optim.AdamW(
    U.param_groups(model, VISUAL_LR, TEXT_LR, WEIGHT_DECAY, VISUAL_LAYER_DECAY),
    betas=(0.9, 0.98), eps=1e-6)
total_steps = EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, U.cosine_with_warmup(WARMUP_STEPS, total_steps))
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
loss_fn = ClipLoss()
trainable = [p for p in model.parameters() if p.requires_grad]

BEST_PATH = os.path.join(OUT_DIR, "best_model.pt")
LAST_PATH = os.path.join(OUT_DIR, "last_checkpoint.pt")
keep_last = SAVE_TO_DRIVE and RESUME
history, step_log, start_epoch, best = [], [], 0, None

if keep_last and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    history, step_log, start_epoch, best = ck["history"], ck["step_log"], ck["epoch"], ck["best"]
    print(f"Resumed after epoch {start_epoch} (best mean R@1 {best['mean_R@1']:.2f})")

def report(m, epoch):
    return (f"epoch {epoch} | i2t R@1/5/10 {m['i2t_R@1']:.1f}/{m['i2t_R@5']:.1f}/{m['i2t_R@10']:.1f} | "
            f"t2i R@1/5/10 {m['t2i_R@1']:.1f}/{m['t2i_R@5']:.1f}/{m['t2i_R@10']:.1f} | mean R@1 {m['mean_R@1']:.2f}")

def maybe_save_best(m, epoch):
    global best
    if best is None or m["mean_R@1"] > best["mean_R@1"]:
        best = dict(m, epoch=epoch)
        U.save_weights(model, BEST_PATH)
        print(f"  ★ new best → {BEST_PATH}")

if start_epoch == 0:
    m = evaluate(model); m["epoch"] = 0; history.append(m)
    print(report(m, 0), "(before training)")
    maybe_save_best(m, 0)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0, run_loss = time.time(), 0.0
    pbar = tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}")
    for it, (images, text_emb) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        text_emb = text_emb.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            img_f, txt_f, logit_scale = model(images, text_emb)
            loss, acc = loss_fn(img_f, txt_f, logit_scale)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if GRAD_CLIP:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        with torch.no_grad():
            model.logit_scale.clamp_(0, math.log(100))  # as in training/train.py

        run_loss += loss.item()
        if it % 20 == 0:
            gstep = epoch * len(train_loader) + it
            step_log.append({"step": gstep, "loss": loss.item(), "scale": logit_scale.item(),
                             "lr": optimizer.param_groups[-1]["lr"]})
            pbar.set_postfix(loss=f"{loss.item():.3f}", i2t=f"{acc['i2t'].item():.2f}",
                             scale=f"{logit_scale.item():.1f}")

    m = evaluate(model)
    m.update(epoch=epoch + 1, train_loss=run_loss / len(train_loader), minutes=(time.time() - t0) / 60)
    history.append(m)
    print(report(m, epoch + 1), f"| train loss {m['train_loss']:.3f} | {m['minutes']:.1f} min")
    maybe_save_best(m, epoch + 1)
    json.dump({"history": history, "best": best}, open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2)
    if keep_last:
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "history": history, "step_log": step_log, "epoch": epoch + 1, "best": best}, LAST_PATH)

json.dump({"history": history, "best": best}, open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2)
print(f"\nBest: epoch {best['epoch']} | mean R@1 {best['mean_R@1']:.2f} | rsum {best['rsum']:.1f}")

In [ ]:
#@title 9 · Training curves
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if step_log:
    ax[0].plot([s["step"] for s in step_log], [s["loss"] for s in step_log], lw=1)
ax[0].set(title="train loss", xlabel="step")
ep = [h["epoch"] for h in history]
for k in ("i2t_R@1", "t2i_R@1", "mean_R@1"):
    ax[1].plot(ep, [h[k] for h in history], marker="o", label=k)
ax[1].axvline(best["epoch"], ls="--", c="gray", lw=1)
ax[1].set(title="val retrieval (dashed = best)", xlabel="epoch", ylabel="%"); ax[1].legend()
plt.tight_layout()
CURVES_PATH = os.path.join(OUT_DIR, "training_curves.png")
plt.savefig(CURVES_PATH, dpi=120); plt.show()

In [ ]:
#@title 10 · Sanity check: retrieve captions for val images with the best weights
model.load_state_dict(torch.load(BEST_PATH, map_location="cpu"))
model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
    T = model.encode_text(val_emb.to(device).float(), normalize=True).float()
    for i in random.Random(0).sample(range(len(val_set)), 3):
        x, _ = val_set[i]
        f = model.encode_image(x[None].to(device), normalize=True).float()
        top = (f @ T.T).squeeze(0).topk(3).indices.tolist()
        plt.figure(figsize=(3, 3)); plt.imshow(U.load_pil(val_ds[i][IMAGE_COLUMN])); plt.axis("off"); plt.show()
        for r, t in enumerate(top, 1):
            hit = "✓" if int(val_owner[t]) == i else " "
            print(f"  {r}. [{hit}] {val_flat[t]}")

In [ ]:
#@title 11 · Package the best model and download it to your computer
LOAD_EXAMPLE = f"""# Load the fine-tuned LLM2CLIP checkpoint on your own machine
# git clone https://github.com/microsoft/LLM2CLIP
# pip install torch torchvision timm einops ftfy regex transformers huggingface_hub pillow
import torch
import llm2clip_utils as U                      # shipped in this zip
U.install_shims("LLM2CLIP/llm2clip")            # path to the cloned repo's llm2clip/ folder
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = U.build_clip_model("{PRESET}", init_from="local", checkpoint_path="best_model.pt")
model = model.to(device).eval()

# Text side: same frozen LLM that produced the training embeddings
enc = U.LLM2CLIPTextEncoder("{P['llm_id']}", device=device,
                            dtype=torch.bfloat16 if device == "cuda" else torch.float32)
captions = ["a dog running on the beach", "two people riding bikes", "a plate of food"]
with torch.no_grad():
    t = model.encode_text(enc.encode(captions).to(device).float(), normalize=True)
    i = model.encode_image(preprocess(Image.open("your_image.jpg")).unsqueeze(0).to(device), normalize=True)
    print((100 * i @ t.T).softmax(-1))
"""

zip_path = os.path.join(OUT_DIR, f"{RUN_NAME}_best.zip")
config = {k: globals()[k] for k in [
    "PRESET", "INIT_FROM", "DATASET_ID", "DATASET_CONFIG", "DATASET_SPLIT", "IMAGE_COLUMN", "CAPTION_COLUMN",
    "MAX_SAMPLES", "VAL_SIZE", "EPOCHS", "BATCH_SIZE", "VISUAL_LR", "TEXT_LR", "WEIGHT_DECAY",
    "VISUAL_LAYER_DECAY", "WARMUP_STEPS", "TRAIN_LAST_N_VISION_BLOCKS", "SEED"]}
config.update(llm_text_encoder=P["llm_id"], clip_config=P["clip_cfg"])
with zipfile.ZipFile(zip_path, "w") as z:
    z.write(BEST_PATH, "best_model.pt", compress_type=zipfile.ZIP_STORED)
    z.writestr("metrics.json", json.dumps({"best": best, "history": history}, indent=2))
    z.writestr("config.json", json.dumps(config, indent=2))
    z.writestr("load_model.py", LOAD_EXAMPLE)
    z.write("/content/llm2clip_utils.py", "llm2clip_utils.py")
    if os.path.exists(CURVES_PATH):
        z.write(CURVES_PATH, "training_curves.png")
print(f"{zip_path}  ({os.path.getsize(zip_path)/1e9:.2f} GB) — best epoch {best['epoch']}, mean R@1 {best['mean_R@1']:.2f}")

if DOWNLOAD_BEST:
    from google.colab import files
    files.download(zip_path)  # large file: the browser download can take a few minutes

### Notes
* **Time budget (rough):** EVA02-L/14 at 336 px is heavy — expect very roughly 30–60 min per Flickr30k epoch on a T4 and several times faster on an A100; caption embedding with the 1B LLM takes a few minutes. Set `MAX_SAMPLES=2000` for a quick test, or `TRAIN_LAST_N_VISION_BLOCKS=0` to train only the adapter (several times faster).
* **Out of memory:** lower `BATCH_SIZE`, keep `GRAD_CHECKPOINTING=True`, or train only the last N vision blocks.
* **Your own data:** point `DATASET_ID` / `IMAGE_COLUMN` / `CAPTION_COLUMN` at any HF dataset; several caption columns (e.g. short + long captions, like the repo's DreamLIP data) can be comma-separated.
* **Using the result:** unzip, keep `llm2clip_utils.py` next to `load_model.py`, clone the repo, and run `load_model.py`. `best_model.pt` has the same format as Microsoft's released `.pt` files, so it also loads with the repo's own `model.load_state_dict(torch.load(...))`.